# 12 — MLB Totals + Run-Line Model Lab

This notebook starts the next two betting functions without disturbing the existing moneyline pipeline.

It trains:

1. **Total runs regression**: predicts `home_score + away_score`.
2. **Home margin regression**: predicts `home_score - away_score`.
3. Optional **home -1.5 cover classifier** for quick diagnostics.

The production scorer can use the total-runs model against totals markets and the margin model against run-line/spread markets.

In [ ]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor, RandomForestClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

try:
    from xgboost import XGBRegressor, XGBClassifier
    HAS_XGB = True
except Exception as exc:
    HAS_XGB = False
    print("XGBoost not available:", exc)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
# Resolve project root from either notebooks/ or repo root.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "mlb_game_features.parquet"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
assert DATA_PATH.exists(), f"Missing feature parquet: {DATA_PATH}"

In [ ]:
features = pd.read_parquet(DATA_PATH)
features["official_date"] = pd.to_datetime(features["official_date"], errors="coerce")
features["game_datetime_utc"] = pd.to_datetime(features["game_datetime_utc"], utc=True, errors="coerce")

print("Rows:", len(features))
print("Columns:", len(features.columns))
print("All date range:", features["official_date"].min(), "to", features["official_date"].max())

## 1. Create totals and run-line targets

In [ ]:
TARGET_TOTAL_RUNS = "target_total_runs"
TARGET_HOME_MARGIN = "target_home_margin"
TARGET_HOME_COVER_MINUS_1_5 = "target_home_cover_minus_1_5"

completed = features[
    features["home_score"].notna()
    & features["away_score"].notna()
].copy()

completed[TARGET_TOTAL_RUNS] = completed["home_score"].astype(float) + completed["away_score"].astype(float)
completed[TARGET_HOME_MARGIN] = completed["home_score"].astype(float) - completed["away_score"].astype(float)
completed[TARGET_HOME_COVER_MINUS_1_5] = (completed[TARGET_HOME_MARGIN] > 1.5).astype(int)

MIN_TRAIN_DATE = "2023-01-01"
completed = completed[completed["official_date"] >= MIN_TRAIN_DATE].copy()

print("Completed rows:", len(completed))
print("Completed range:", completed["official_date"].min(), "to", completed["official_date"].max())
print("Mean total runs:", completed[TARGET_TOTAL_RUNS].mean())
print("Mean home margin:", completed[TARGET_HOME_MARGIN].mean())
print("Home -1.5 cover rate:", completed[TARGET_HOME_COVER_MINUS_1_5].mean())

## 2. Leakage-safe feature filtering

For totals and run-line models, postgame scores and result columns are leaks. We also avoid market/result/recommendation columns as model inputs.

In [ ]:
LEAKY_COLS_EXACT = {
    "home_score", "away_score", "diff_score", "home_margin", "target_home_win",
    TARGET_TOTAL_RUNS, TARGET_HOME_MARGIN, TARGET_HOME_COVER_MINUS_1_5,
}

METADATA_COLS = {
    "game_pk", "run_id", "scored_at_utc", "official_date", "official_date_dt", "game_datetime_utc",
    "home_team_name", "away_team_name", "home_team_id", "away_team_id", "venue_name",
    "abstract_state", "detailed_state",
}

LEAKY_PATTERNS = [
    "winner", "winning", "losing", "final", "result", "outcome", "actual", "post_",
    "recommended", "edge", "kelly", "suggested", "market_", "moneyline", "price", "odds",
    "home_score", "away_score", "margin", "target_",
]

def is_probably_leaky_feature(col: str) -> bool:
    c = col.lower()
    if c in LEAKY_COLS_EXACT or c in METADATA_COLS:
        return True
    if c.endswith("_id"):
        return True
    return any(p in c for p in LEAKY_PATTERNS)

candidate_feature_cols = [
    c for c in completed.columns
    if c not in METADATA_COLS
    and c not in LEAKY_COLS_EXACT
    and pd.api.types.is_numeric_dtype(completed[c])
]

clean_feature_cols = [c for c in candidate_feature_cols if not is_probably_leaky_feature(c)]

print("Candidate feature count:", len(candidate_feature_cols))
print("Clean feature count:", len(clean_feature_cols))

removed = sorted(set(candidate_feature_cols) - set(clean_feature_cols))
display(pd.Series(removed, name="removed_features").head(100))

## 3. Build focused feature families

In [ ]:
def cols_containing_any(cols, terms):
    terms = [t.lower() for t in terms]
    return [c for c in cols if any(t in c.lower() for t in terms)]

def contains_any(col, terms):
    c = col.lower()
    return any(t in c for t in terms)

PITCHMIX_TERMS = ["pitchmix", "pitch_mix", "pitch_type", "pitchtype", "coarse_pitch", "arsenal", "matchup"]
BULLPEN_AVAIL_TERMS = ["bullpen_avail", "availability", "fatigue", "bullpen_pitches_last", "sc_bullpen_pitches_last", "diff_bullpen_pitches_last", "diff_sc_bullpen_pitches_last", "relievers_used", "back_to_back"]
BULLPEN_QUALITY_TERMS = ["bullpen_sc", "diff_bullpen_sc", "home_bullpen_sc", "away_bullpen_sc", "sc_bullpen"]
STATCAST_TERMS = ["statcast", "_sc_", "sc_", "batted_ball", "avg_ev", "distance", "hard_hit", "barrel"]

all_numeric_cols = [c for c in clean_feature_cols if c in completed.columns and pd.api.types.is_numeric_dtype(completed[c])]
statcast_cols = cols_containing_any(all_numeric_cols, STATCAST_TERMS)
pitchmix_cols = cols_containing_any(all_numeric_cols, PITCHMIX_TERMS)
bullpen_availability_cols = cols_containing_any(all_numeric_cols, BULLPEN_AVAIL_TERMS)
bullpen_quality_cols = cols_containing_any(all_numeric_cols, BULLPEN_QUALITY_TERMS)
old_statcast_cols = [c for c in statcast_cols if not contains_any(c, PITCHMIX_TERMS + BULLPEN_AVAIL_TERMS)]

feature_sets = {
    "old_statcast_no_pitchmix_bullpen": sorted(set(old_statcast_cols)),
    "true_pitchmix": sorted(set(pitchmix_cols)),
    "true_bullpen_availability": sorted(set(bullpen_availability_cols)),
    "pitchmix_plus_bullpen": sorted(set(pitchmix_cols + bullpen_quality_cols + bullpen_availability_cols)),
    "old_statcast_plus_pitchmix_bullpen": sorted(set(old_statcast_cols + pitchmix_cols + bullpen_availability_cols)),
    "all_clean": sorted(set(all_numeric_cols)),
}

feature_family_summary = pd.DataFrame([{"feature_family": k, "feature_count": len(v)} for k, v in feature_sets.items()]).sort_values("feature_count", ascending=False)
display(feature_family_summary)

MODEL_FEATURE_FAMILIES = [
    "old_statcast_no_pitchmix_bullpen",
    "pitchmix_plus_bullpen",
    "old_statcast_plus_pitchmix_bullpen",
    "all_clean",
]

## 4. Chronological train/test split

In [ ]:
def chronological_split(df: pd.DataFrame, test_frac: float = 0.20):
    df = df.sort_values(["official_date", "game_datetime_utc", "game_pk"]).reset_index(drop=True)
    split_idx = int(len(df) * (1 - test_frac))
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

train_df, test_df = chronological_split(completed, test_frac=0.20)
print("Train rows:", len(train_df), train_df["official_date"].min(), "to", train_df["official_date"].max())
print("Test rows:", len(test_df), test_df["official_date"].min(), "to", test_df["official_date"].max())

## 5. Model factories

In [ ]:
def make_regression_models():
    models = {
        "random_forest": RandomForestRegressor(n_estimators=400, min_samples_leaf=12, random_state=42, n_jobs=-1),
        "extra_trees": ExtraTreesRegressor(n_estimators=500, min_samples_leaf=10, random_state=42, n_jobs=-1),
        "hist_gbdt": HistGradientBoostingRegressor(max_iter=350, learning_rate=0.035, l2_regularization=2.0, random_state=42),
    }
    if HAS_XGB:
        models["xgboost"] = XGBRegressor(
            n_estimators=700,
            max_depth=2,
            learning_rate=0.025,
            subsample=0.85,
            colsample_bytree=0.70,
            min_child_weight=20,
            reg_lambda=8.0,
            reg_alpha=0.25,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1,
        )
    return models


def make_classifier_models():
    models = {
        "random_forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=12, random_state=42, n_jobs=-1),
        "extra_trees": ExtraTreesClassifier(n_estimators=500, min_samples_leaf=10, random_state=42, n_jobs=-1),
        "logit_l2": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=3000, C=0.25, solver="lbfgs")),
        ]),
    }
    if HAS_XGB:
        models["xgboost"] = XGBClassifier(
            n_estimators=700,
            max_depth=2,
            learning_rate=0.025,
            subsample=0.85,
            colsample_bytree=0.70,
            min_child_weight=20,
            reg_lambda=8.0,
            reg_alpha=0.25,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        )
    return models


def wrap_tree_model(estimator):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", estimator),
    ])

## 6. Train total-runs regression models

In [ ]:
def evaluate_regression(y_true, pred):
    return {
        "mae": float(mean_absolute_error(y_true, pred)),
        "rmse": float(mean_squared_error(y_true, pred, squared=False)),
        "r2": float(r2_score(y_true, pred)),
        "avg_pred": float(np.mean(pred)),
        "actual_mean": float(np.mean(y_true)),
    }


totals_results = []
totals_fitted = {}
totals_preds = {}

y_train_total = train_df[TARGET_TOTAL_RUNS].astype(float)
y_test_total = test_df[TARGET_TOTAL_RUNS].astype(float)

for family in MODEL_FEATURE_FAMILIES:
    cols = [c for c in feature_sets[family] if c in train_df.columns]
    if not cols:
        continue
    for model_name, base_model in make_regression_models().items():
        estimator = wrap_tree_model(base_model)
        full_name = f"{family}__{model_name}"
        print("Training totals:", full_name, "features", len(cols))
        estimator.fit(train_df[cols], y_train_total)
        pred = estimator.predict(test_df[cols])
        metrics = evaluate_regression(y_test_total, pred)
        totals_results.append({"model_name": full_name, "feature_count": len(cols), "n_test": len(y_test_total), **metrics})
        totals_fitted[full_name] = (estimator, cols)
        totals_preds[full_name] = pred

totals_results_df = pd.DataFrame(totals_results).sort_values(["mae", "rmse"]).reset_index(drop=True)
display(totals_results_df.head(20))

## 7. Train home-margin regression models

In [ ]:
margin_results = []
margin_fitted = {}
margin_preds = {}

y_train_margin = train_df[TARGET_HOME_MARGIN].astype(float)
y_test_margin = test_df[TARGET_HOME_MARGIN].astype(float)

for family in MODEL_FEATURE_FAMILIES:
    cols = [c for c in feature_sets[family] if c in train_df.columns]
    if not cols:
        continue
    for model_name, base_model in make_regression_models().items():
        estimator = wrap_tree_model(base_model)
        full_name = f"{family}__{model_name}"
        print("Training margin:", full_name, "features", len(cols))
        estimator.fit(train_df[cols], y_train_margin)
        pred = estimator.predict(test_df[cols])
        metrics = evaluate_regression(y_test_margin, pred)
        margin_results.append({"model_name": full_name, "feature_count": len(cols), "n_test": len(y_test_margin), **metrics})
        margin_fitted[full_name] = (estimator, cols)
        margin_preds[full_name] = pred

margin_results_df = pd.DataFrame(margin_results).sort_values(["mae", "rmse"]).reset_index(drop=True)
display(margin_results_df.head(20))

## 8. Optional run-line cover classifier diagnostic

In [ ]:
cover_results = []
cover_fitted = {}
cover_preds = {}

y_train_cover = train_df[TARGET_HOME_COVER_MINUS_1_5].astype(int)
y_test_cover = test_df[TARGET_HOME_COVER_MINUS_1_5].astype(int)

for family in MODEL_FEATURE_FAMILIES:
    cols = [c for c in feature_sets[family] if c in train_df.columns]
    if not cols:
        continue
    for model_name, model in make_classifier_models().items():
        estimator = model if isinstance(model, Pipeline) else wrap_tree_model(model)
        full_name = f"{family}__{model_name}"
        print("Training cover classifier:", full_name, "features", len(cols))
        estimator.fit(train_df[cols], y_train_cover)
        p = estimator.predict_proba(test_df[cols])[:, 1]
        cover_results.append({
            "model_name": full_name,
            "feature_count": len(cols),
            "n_test": len(y_test_cover),
            "avg_pred": float(np.mean(p)),
            "actual_rate": float(y_test_cover.mean()),
            "log_loss": float(log_loss(y_test_cover, p)),
            "brier": float(brier_score_loss(y_test_cover, p)),
            "roc_auc": float(roc_auc_score(y_test_cover, p)),
            "accuracy_50pct": float(accuracy_score(y_test_cover, p >= 0.5)),
        })
        cover_fitted[full_name] = (estimator, cols)
        cover_preds[full_name] = p

cover_results_df = pd.DataFrame(cover_results).sort_values(["log_loss", "brier"]).reset_index(drop=True)
display(cover_results_df.head(20))

## 9. Residual diagnostics for champion regressors

In [ ]:
TOTALS_CHAMPION_NAME = totals_results_df.iloc[0]["model_name"]
MARGIN_CHAMPION_NAME = margin_results_df.iloc[0]["model_name"]

print("Totals champion:", TOTALS_CHAMPION_NAME)
display(totals_results_df[totals_results_df["model_name"].eq(TOTALS_CHAMPION_NAME)])

print("Margin champion:", MARGIN_CHAMPION_NAME)
display(margin_results_df[margin_results_df["model_name"].eq(MARGIN_CHAMPION_NAME)])

total_resid = y_test_total.values - totals_preds[TOTALS_CHAMPION_NAME]
margin_resid = y_test_margin.values - margin_preds[MARGIN_CHAMPION_NAME]

resid_summary = pd.DataFrame([
    {"target": "total_runs", "residual_mean": total_resid.mean(), "residual_std": total_resid.std(ddof=1), "p10": np.quantile(total_resid, 0.10), "p50": np.quantile(total_resid, 0.50), "p90": np.quantile(total_resid, 0.90)},
    {"target": "home_margin", "residual_mean": margin_resid.mean(), "residual_std": margin_resid.std(ddof=1), "p10": np.quantile(margin_resid, 0.10), "p50": np.quantile(margin_resid, 0.50), "p90": np.quantile(margin_resid, 0.90)},
])
display(resid_summary)

## 10. Export champion model bundles

Set the approval flags to `True` only after reviewing metrics.

The scoring script uses `residual_std` to turn a predicted total or margin into over/under and run-line cover probabilities.

In [ ]:
APPROVE_TOTALS_EXPORT = False
APPROVE_MARGIN_EXPORT = False

if APPROVE_TOTALS_EXPORT:
    estimator, cols = totals_fitted[TOTALS_CHAMPION_NAME]
    pred = totals_preds[TOTALS_CHAMPION_NAME]
    residual = y_test_total.values - pred
    bundle = {
        "model_name": TOTALS_CHAMPION_NAME,
        "model_kind": "regression",
        "market": "totals",
        "target_col": TARGET_TOTAL_RUNS,
        "model": estimator,
        "feature_cols": cols,
        "created_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "min_train_date": MIN_TRAIN_DATE,
        "residual_mean": float(np.mean(residual)),
        "residual_std": float(np.std(residual, ddof=1)),
        "metrics": totals_results_df[totals_results_df["model_name"].eq(TOTALS_CHAMPION_NAME)].iloc[0].to_dict(),
    }
    path = MODEL_DIR / "mlb_total_runs_champion.joblib"
    joblib.dump(bundle, path)
    print("Exported", path)

if APPROVE_MARGIN_EXPORT:
    estimator, cols = margin_fitted[MARGIN_CHAMPION_NAME]
    pred = margin_preds[MARGIN_CHAMPION_NAME]
    residual = y_test_margin.values - pred
    bundle = {
        "model_name": MARGIN_CHAMPION_NAME,
        "model_kind": "regression",
        "market": "runline",
        "target_col": TARGET_HOME_MARGIN,
        "model": estimator,
        "feature_cols": cols,
        "created_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "min_train_date": MIN_TRAIN_DATE,
        "residual_mean": float(np.mean(residual)),
        "residual_std": float(np.std(residual, ddof=1)),
        "metrics": margin_results_df[margin_results_df["model_name"].eq(MARGIN_CHAMPION_NAME)].iloc[0].to_dict(),
    }
    path = MODEL_DIR / "mlb_home_margin_champion.joblib"
    joblib.dump(bundle, path)
    print("Exported", path)

print("APPROVE_TOTALS_EXPORT:", APPROVE_TOTALS_EXPORT)
print("APPROVE_MARGIN_EXPORT:", APPROVE_MARGIN_EXPORT)

## 11. Odds prerequisite check for production

Before totals/run-line scoring can generate recommendations, the Odds API table must contain `totals` and `spreads` markets, not just `h2h`.

Run this locally/Cloud Shell after a daily odds fetch:

```sql
SELECT market_key, COUNT(*) FROM odds_snapshots GROUP BY market_key;
```

If only `h2h` appears, update the odds fetch job/script to request `h2h,spreads,totals`.

In [ ]:
# Optional local DB market inspection.
import sqlite3
DB_PATH = PROJECT_ROOT / "data" / "odds.db"
if DB_PATH.exists() and DB_PATH.stat().st_size > 0:
    conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
    try:
        markets = pd.read_sql_query("SELECT market_key, COUNT(*) AS rows FROM odds_snapshots GROUP BY market_key ORDER BY rows DESC", conn)
        display(markets)
    finally:
        conn.close()
else:
    print("No local odds.db found; skip market inspection.")